# المرحلة الثانية: النماذج الأساسية

نحول النصوص إلى تمثيل رقمي باستخدام **Character TF-IDF**، ثم نقارن بين:

- Logistic Regression
- LinearSVC

النماذج المدربة محفوظة مسبقًا في مجلد `models/`. خلية التدريب اختيارية لأنها قد تستغرق وقتًا.


## لماذا Character TF-IDF؟

نستخدم مقاطع من 3 إلى 5 أحرف بدل الكلمات الكاملة. هذا مناسب للهجات العربية لأنه:

- يتحمل بعض الأخطاء الإملائية.
- يلتقط أجزاء الكلمات المميزة للهجة.
- لا يحتاج إلى محلل صرفي.
- يناسب النصوص القصيرة.


In [ ]:
# Path للتعامل مع مسارات الملفات
from pathlib import Path
# json لقراءة ملفات JSON وJSONL
import json
# pandas لعرض البيانات والإحصاءات في جداول
import pandas as pd
# يحول النص إلى أوزان رقمية باستخدام TF-IDF على مستوى الحروف
from sklearn.feature_extraction.text import TfidfVectorizer

# نحدد جذر المشروع الحالي
ROOT = Path.cwd()
# نحدد مجلد البيانات المعالجة
PROCESSED = ROOT / "data" / "processed"

# دالة لتحميل تقسيم من ملفات JSONL
def load_rows(split):
# نفتح ملف التقسيم المطلوب
    with (PROCESSED / f"{split}.jsonl").open(encoding="utf-8") as handle:
# نحول الأسطر غير الفارغة إلى سجلات Python
        return [json.loads(line) for line in handle if line.strip()]

train_rows = load_rows("train")
test_rows = load_rows("test")
print("عدد سجلات التدريب:", len(train_rows))
print("عدد سجلات الاختبار:", len(test_rows))


## تجهيز مهمة تصنيف المنطقة

In [ ]:
# نحتفظ بالسجلات ذات النص والتصنيف المعروف
def task_rows(rows, label):
    return [row for row in rows if row.get("text") and row.get(label) != "Unknown"]

# نجهز بيانات تدريب المنطقة
region_train = task_rows(train_rows, "region")
# نجهز بيانات اختبار المنطقة
region_test = task_rows(test_rows, "region")
# نستخرج نصوص التدريب
x_train = [row["text"] for row in region_train]
# نستخرج نصوص الاختبار
x_test = [row["text"] for row in region_test]
# نستخرج تسميات التدريب
y_train = [row["region"] for row in region_train]
# نستخرج تسميات الاختبار
y_test = [row["region"] for row in region_test]

# ننشئ محول Character TF-IDF
vectorizer = TfidfVectorizer(
# نستخدم مقاطع أحرف بدل الكلمات
    analyzer="char",
# نستخدم مقاطع من 3 إلى 5 أحرف
    ngram_range=(3, 5),
# نتجاهل المقاطع النادرة جدًا
    min_df=2,
# نحدد الحد الأقصى للميزات
    max_features=300_000,
# نستخدم مقياسًا فرعيًا لتأثير التكرار
    sublinear_tf=True,
# نستخدم نوعًا رقميًا أخف
    dtype="float32",
)
# نتعلم الميزات من التدريب ونحوله إلى مصفوفة
x_train_matrix = vectorizer.fit_transform(x_train)
# نحول الاختبار بنفس قاموس الميزات
x_test_matrix = vectorizer.transform(x_test)
print("شكل مصفوفة التدريب:", x_train_matrix.shape)


## التدريب الاختياري

اجعل `RUN_TRAINING = True` إذا أردت إعادة التدريب. في العرض التعليمي يكفي تحميل النتائج
المحفوظة مسبقًا لتجنب استهلاك وقت وذاكرة كبيرين.


In [ ]:
# نستخدم النتائج المحفوظة افتراضيًا
RUN_TRAINING = False

# نعيد التدريب فقط إذا غُير هذا الخيار إلى True
if RUN_TRAINING:
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.metrics import accuracy_score, f1_score

    classifiers = {
        "Logistic Regression": LogisticRegression(
            max_iter=100, class_weight="balanced", solver="saga", n_jobs=-1
        ),
        "LinearSVC": LinearSVC(class_weight="balanced"),
    }
    results = []
    for name, classifier in classifiers.items():
# ندرب المصنف على النصوص والتسميات
        classifier.fit(x_train_matrix, y_train)
# نتنبأ بفئات نصوص الاختبار
        prediction = classifier.predict(x_test_matrix)
        results.append({
            "النموذج": name,
            "Accuracy": accuracy_score(y_test, prediction),
            "Macro-F1": f1_score(y_test, prediction, average="macro"),
        })
    pd.DataFrame(results)
else:
# نقرأ نتائج التدريب المحفوظة
    metrics = json.loads((ROOT / "results" / "baseline_metrics.json").read_text(encoding="utf-8"))
    rows = []
    for name, item in metrics["tasks"]["region"]["models"].items():
        rows.append({
            "النموذج": name,
            "Accuracy": item["test"]["accuracy"],
            "Macro-F1": item["test"]["f1_macro"],
        })
    pd.DataFrame(rows)


## تفسير النتيجة

حقق LinearSVC أفضل أداء إقليمي:

- Accuracy ≈ 87.57%
- Macro-F1 ≈ 81.93%

نستخدم Macro-F1 لأن البيانات غير متوازنة تمامًا بين المناطق.


In [ ]:
# نقرأ نتائج التدريب المحفوظة
metrics = json.loads((ROOT / "results" / "baseline_metrics.json").read_text(encoding="utf-8"))
region_test_results = {
    name: {
        "Accuracy": model["test"]["accuracy"],
        "Macro-F1": model["test"]["f1_macro"],
    }
    for name, model in metrics["tasks"]["region"]["models"].items()
}
pd.DataFrame(region_test_results).T
